In [11]:
import pandas as pd
import numpy as np
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')


In [12]:
# 캐나다 달러와 US 달러 통일
train['amount_in_usd'] = train.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)
test['amount_in_usd'] = train.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)

In [13]:
#total_open_amount의 분포가 매우 치우쳐져 있기 때문에, 로그 변환을 통해 분포 완화
train['log_total_open_amount'] = np.log1p(train['total_open_amount'])
test['log_total_open_amount'] = np.log1p(test['total_open_amount'])

In [14]:
# 결제 수단의 등장 횟수가 30회 이하인 경우 Other 클래스로 변환
counts = train["cust_payment_terms"].value_counts()
train["cust_payment_terms_grp"] = train["cust_payment_terms"].where(
    train["cust_payment_terms"].map(counts) > 30,
    "Other"
)
test["cust_payment_terms_grp"] = test["cust_payment_terms"].where(
    test["cust_payment_terms"].map(counts) > 30,
    "Other"
)

In [15]:
# 송장이 생성된 기준 날짜의 월, 일, 요일 추출
train['baseline_create_date'] = pd.to_datetime(train['baseline_create_date'], format='%Y%m%d', errors='coerce')
test['baseline_create_date'] = pd.to_datetime(test['baseline_create_date'], format='%Y%m%d', errors='coerce')

train['baseline_month'] = train['baseline_create_date'].dt.month
train['baseline_day'] = train['baseline_create_date'].dt.day
train['baseline_dayofweek'] = train['baseline_create_date'].dt.dayofweek
test['baseline_month'] = test['baseline_create_date'].dt.month
test['baseline_day'] = test['baseline_create_date'].dt.day
test['baseline_dayofweek'] = test['baseline_create_date'].dt.dayofweek


In [16]:
# 송장 생성일과 마감일 사이의 기간 추가
train['due_in_date'] = pd.to_datetime(train['due_in_date'], format='%Y%m%d', errors='coerce')
test['due_in_date'] = pd.to_datetime(test['due_in_date'], format='%Y%m%d', errors='coerce')

train['Allowed_Pay_Days'] = (train['due_in_date'] - train['baseline_create_date']).dt.days
test['Allowed_Pay_Days'] = (test['due_in_date'] - test['baseline_create_date']).dt.days

In [17]:
# cust_number 자릿수 맞추기
def clean_cust_number(df):
    df = df.copy()

    cust_number_stripped = df["cust_number"].astype(str).str.strip()
    is_9_digit = cust_number_stripped.str.fullmatch(r"\d{9}")

    df["cust_number"] = cust_number_stripped.where(
        ~is_9_digit,
        cust_number_stripped.str.zfill(10)
    )

    return df

train = clean_cust_number(train)
test = clean_cust_number(test)

In [18]:
drop_cols = ['doc_id', 'invoice_currency', 'document type', 'area_business', 'isOpen', 'invoice_id','document_create_date', 'document_create_date.1', 'posting_date']
train.drop(columns = drop_cols, inplace=True)
test.drop(columns = drop_cols, inplace=True)


In [22]:
train.to_csv('../data/train_cleaned.csv', index=False)
test.to_csv('../data/test_cleaned.csv', index=False)